# ARC_ATLAS_Test_v4

Evaluate trained checkpoints on local splits and downsampled variants.

In [ ]:

from pathlib import Path
import importlib.util, json
import tensorflow as tf
import numpy as np
PROJECT_ROOT = Path(__file__).resolve().parent
SRC = PROJECT_ROOT / 'src' / 'training_v2.py'

# Pick run (prefer runs/latest symlink)
LATEST_LINK = PROJECT_ROOT / 'runs' / 'latest'
if LATEST_LINK.exists():
    RUN = LATEST_LINK.resolve()
else:
    RUNS = sorted((PROJECT_ROOT/'runs').glob('20*'))
    if not RUNS:
        raise SystemExit('No runs found; train first.')
    RUN = RUNS[-1]

WEIGHTS = (RUN/'callbacks'/'best_model_dynamic.weights.h5')
if not WEIGHTS.exists():
    # fallback: latest_best copy
    alt = PROJECT_ROOT / 'runs' / 'latest_best.weights.h5'
    WEIGHTS = alt if alt.exists() else WEIGHTS
print('Using run:', RUN)
print('Weights:', WEIGHTS)

spec = importlib.util.spec_from_file_location('seg', SRC)
seg = importlib.util.module_from_spec(spec); spec.loader.exec_module(seg)
cfg_json = json.load(open(RUN/'models'/'config.json'))
# Force local data paths
cfg_json['DATA_DIR'] = str(PROJECT_ROOT/'data'/'splits'/'50_25_25'/'train_hires')
cfg_json['IMAGES_DIR'] = str(PROJECT_ROOT/'data'/'splits'/'50_25_25'/'train_hires'/'t1')
cfg_json['MASKS_DIR'] = str(PROJECT_ROOT/'data'/'splits'/'50_25_25'/'train_hires'/'masks')
cfg = seg.DynamicTrainingConfig(**cfg_json)
cfg.MODEL_DIR = RUN/'models'
model = seg.build_model_for_inference(cfg, weights_path=str(WEIGHTS))

# quick zero test
x0 = np.zeros((1,*cfg.INPUT_SHAPE), np.float32)
p0 = model.predict(x0, verbose=0)[0,...,0]
print('blank mean', float(p0.mean()), 'max', float(p0.max()))


## (Optional) Evaluate on downsampled sets
Use helper functions from `src/downsampling/*.py` to generate degraded test sets in `data/downsampled/`, then load them with the training loader for metrics.